# Асинхронный веб-краулер

### Установка зависимостей

In [1]:
%pip install aiohttp aiofiles beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [2]:
import asyncio
import aiohttp
import logging
import time
import json
import itertools
import re
import aiofiles

from collections import Counter
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from contextlib import asynccontextmanager, suppress


### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

## Класс HTMLParser

In [4]:
class HTMLParser:
    def empty_result(self, url: str) -> dict:
        return {
            "url": url,
            "title": "",
            "text": "",
            "links": [],
            "metadata": {},
            "images": [],
            "headings": {},
            "tables": [],
            "lists": [],
        }

    async def parse_html(self, html: str, url: str) -> dict:
        result = self.empty_result(url)
        try:
            soup = await asyncio.to_thread(BeautifulSoup, html, "lxml")
        except Exception as e:
            logger.warning("Не удалось разобрать HTML (%s): %s", type(e).__name__, url)
            return result

        extractors = {
            "metadata": lambda: self.extract_metadata(soup),
            "text": lambda: self.extract_text(soup),
            "links": lambda: self.extract_links(soup, url),
            "images": lambda: self.extract_images(soup, url),
            "headings": lambda: self.extract_headings(soup),
            "tables": lambda: self.extract_tables(soup),
            "lists": lambda: self.extract_lists(soup),
        }
        for field, extract in extractors.items():
            try:
                result[field] = extract()
            except Exception as e:
                logger.warning(
                    "Ошибка при извлечении %s (%s): %s", field, type(e).__name__, url
                )

        result["title"] = result["metadata"].get("title", "")
        return result

    def extract_links(self, soup: BeautifulSoup, base_url: str) -> list[str]:
        links = []
        for tag in soup.find_all("a", href=True):
            absolute_url = urljoin(base_url, tag["href"])
            if self.is_valid_url(absolute_url):
                links.append(absolute_url)
        return links

    def is_valid_url(self, url: str) -> bool:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and bool(parsed.netloc)

    def extract_text(self, soup: BeautifulSoup, selector: str = None) -> str:
        root = soup.select_one(selector) if selector else soup
        if root is None:
            return ""
        return root.get_text(separator=" ", strip=True)

    def extract_metadata(self, soup: BeautifulSoup) -> dict:
        def meta_content(name: str) -> str:
            tag = soup.find("meta", attrs={"name": name})
            return tag.get("content", "") if tag else ""

        title = soup.title.get_text(strip=True) if soup.title else ""
        return {
            "title": title,
            "description": meta_content("description"),
            "keywords": meta_content("keywords"),
        }
    
    def extract_images(self, soup: BeautifulSoup, base_url: str) -> list[dict]:
        images = []
        for tag in soup.find_all("img", src=True):
            images.append({
                "src": urljoin(base_url, tag["src"]),
                "alt": tag.get("alt", ""),
            })
        return images

    def extract_headings(self, soup: BeautifulSoup) -> dict[str, list[str]]:
        headings = {}
        for level in ("h1", "h2", "h3"):
            headings[level] = [
                tag.get_text(strip=True) for tag in soup.find_all(level)
            ]
        return headings

    def extract_tables(self, soup: BeautifulSoup) -> list[list[list[str]]]:
        tables = []
        for table in soup.find_all("table"):
            rows = []
            for tr in table.find_all("tr"):
                cells = [
                    cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])
                ]
                if cells:
                    rows.append(cells)
            tables.append(rows)
        return tables

    def extract_lists(self, soup: BeautifulSoup) -> list[dict]:
        lists = []
        for tag in soup.find_all(["ul", "ol"]):
            items = [
                li.get_text(strip=True) for li in tag.find_all("li", recursive=False)
            ]
            lists.append({"type": tag.name, "items": items})
        return lists

## Класс CrawlerQueue

In [5]:
class CrawlerQueue:
    def __init__(self):
        self.queue = asyncio.PriorityQueue()
        self.counter = itertools.count()
        self.seen: set[str] = set()
        self.processed: set[str] = set()
        self.failed: dict[str, str] = {}

    def add_url(self, url: str, priority: int = 0):
        if url in self.seen:
            return
        self.seen.add(url)
        self.queue.put_nowait((-priority, next(self.counter), url))

    async def get_next(self) -> str | None:
        if self.queue.empty():
            return None
        _, _, url = await self.queue.get()
        return url

    def mark_processed(self, url: str):
        self.processed.add(url)

    def mark_failed(self, url: str, error: str):
        self.failed[url] = error

    def get_stats(self) -> dict:
        return {
            "queued": self.queue.qsize(),
            "processed": len(self.processed),
            "failed": len(self.failed),
            "total_seen": len(self.seen),
        }

## Класс SemaphoreManager

In [6]:
class SemaphoreManager:
    def __init__(self, global_limit: int = 10, per_domain_limit: int = 2):
        self.global_semaphore = asyncio.Semaphore(global_limit)
        self.per_domain_limit = per_domain_limit
        self.domain_semaphores: dict[str, asyncio.Semaphore] = {}
        self.active_total = 0
        self.active_by_domain: dict[str, int] = {}

    @asynccontextmanager
    async def acquire(self, url: str):
        domain = urlparse(url).netloc
        if domain not in self.domain_semaphores:
            self.domain_semaphores[domain] = asyncio.Semaphore(self.per_domain_limit)

        async with self.domain_semaphores[domain]:
            async with self.global_semaphore:
                self.active_total += 1
                self.active_by_domain[domain] = self.active_by_domain.get(domain, 0) + 1
                try:
                    yield
                finally:
                    self.active_total -= 1
                    self.active_by_domain[domain] -= 1

    def get_active(self) -> dict:
        return {
            "total": self.active_total,
            "by_domain": {
                domain: count
                for domain, count in self.active_by_domain.items()
                if count > 0
            },
        }

## Класс AsyncCrawler

In [7]:
class AsyncCrawler:
    def __init__(
        self,
        max_concurrent: int = 10,
        max_depth: int = 2,
        per_domain_limit: int | None = None,
        connect_timeout: float = 5,
        read_timeout: float = 10,
    ):
        self.max_concurrent = max_concurrent
        self.max_depth = max_depth
        self.semaphores = SemaphoreManager(
            global_limit=max_concurrent,
            per_domain_limit=per_domain_limit or max_concurrent,
        )
        self.parser = HTMLParser()

        self.queue = CrawlerQueue()
        self.visited_urls: set[str] = set()
        self.failed_urls: dict[str, str] = {}
        self.processed_urls: dict[str, dict] = {}
        self.url_depths: dict[str, int] = {}
        self.in_progress = 0
        self.same_domain_only = False
        self.allowed_domains: set[str] = set()
        self.exclude_patterns: list[re.Pattern] = []
        self.include_patterns: list[re.Pattern] = []

        timeout = aiohttp.ClientTimeout(
            connect=connect_timeout,
            sock_read=read_timeout,
        )
        # используется системный поиск адресов, чтобы краулер работал через VPN
        connector = aiohttp.TCPConnector(
            limit=max_concurrent,
            resolver=aiohttp.ThreadedResolver(),
        )

        self.session = aiohttp.ClientSession(
            timeout=timeout,
            connector=connector,
        )

    async def fetch_url(self, url: str) -> str | None:
        async with self.semaphores.acquire(url):
            logger.info("Начало загрузки: %s", url)
            try:
                async with self.session.get(url) as response:
                    response.raise_for_status()
                    text = await response.text()
                    logger.info(
                        "Успешно: %s (статус %s, %d символов)",
                        url, response.status, len(text),
                    )
                    return text
            except aiohttp.ClientResponseError as e:
                logger.error("%s, статус %s: %s", type(e).__name__, e.status, url)
                self.failed_urls[url] = f"{type(e).__name__} {e.status}"
            except asyncio.TimeoutError as e:
                logger.error("%s, таймаут: %s", type(e).__name__, url)
                self.failed_urls[url] = type(e).__name__
            except aiohttp.ClientError as e:
                logger.error("%s, сетевая ошибка: %s", type(e).__name__, url)
                self.failed_urls[url] = type(e).__name__
            return None

    async def fetch_urls(self, urls: list[str]) -> dict[str, str]:
        tasks = [self.fetch_url(url) for url in urls]
        pages = await asyncio.gather(*tasks)
        return {url: page for url, page in zip(urls, pages) if page is not None}

    async def fetch_and_parse(self, url: str) -> dict:
        html = await self.fetch_url(url)
        if html is None:
            return self.parser.empty_result(url)
        return await self.parser.parse_html(html, url)

    async def crawl(
        self,
        start_urls: list[str],
        max_pages: int = 100,
        same_domain_only: bool = False,
        exclude_patterns: list[str] | None = None,
        include_patterns: list[str] | None = None,
    ) -> dict[str, dict]:
        self.queue = CrawlerQueue()
        self.visited_urls = set()
        self.failed_urls = {}
        self.processed_urls = {}
        self.url_depths = {}
        self.in_progress = 0

        self.same_domain_only = same_domain_only
        self.allowed_domains = {urlparse(url).netloc for url in start_urls}
        self.exclude_patterns = [re.compile(p) for p in exclude_patterns or []]
        self.include_patterns = [re.compile(p) for p in include_patterns or []]

        for url in start_urls:
            url, _ = urldefrag(url)
            self.url_depths[url] = 0
            self.queue.add_url(url, priority=0)

        start_time = time.perf_counter()
        progress = asyncio.create_task(self.report_progress(start_time))
        workers = [
            asyncio.create_task(self.worker(max_pages))
            for _ in range(self.max_concurrent)
        ]
        try:
            await asyncio.gather(*workers)
        finally:
            progress.cancel()
            with suppress(asyncio.CancelledError):
                await progress
            self.print_progress(start_time)
        return self.processed_urls

    def is_allowed(self, url: str) -> bool:
        if self.same_domain_only and urlparse(url).netloc not in self.allowed_domains:
            return False
        if any(pattern.search(url) for pattern in self.exclude_patterns):
            return False
        if self.include_patterns and not any(
            pattern.search(url) for pattern in self.include_patterns
        ):
            return False
        return True

    async def report_progress(self, start_time: float, interval: float = 1.0):
        while True:
            await asyncio.sleep(interval)
            self.print_progress(start_time)

    def print_progress(self, start_time: float):
        stats = self.queue.get_stats()
        elapsed = time.perf_counter() - start_time
        speed = stats["processed"] / elapsed if elapsed > 0 else 0
        print(
            f"Обработано: {stats['processed']} | "
            f"В очереди: {stats['queued']} | "
            f"Ошибок: {stats['failed']} | "
            f"Скорость: {speed:.1f} стр/сек"
        )

    async def worker(self, max_pages: int):
        while len(self.visited_urls) < max_pages:
            url = await self.queue.get_next()
            if url is None:
                if self.in_progress == 0:
                    break
                await asyncio.sleep(0.1)
                continue

            self.visited_urls.add(url)
            self.in_progress += 1
            try:
                await self.process_url(url)
            finally:
                self.in_progress -= 1

    async def process_url(self, url: str):
        depth = self.url_depths[url]
        page = await self.fetch_and_parse(url)

        if url in self.failed_urls:
            self.queue.mark_failed(url, self.failed_urls[url])
            return

        self.processed_urls[url] = page
        self.queue.mark_processed(url)

        if depth >= self.max_depth:
            return
        for link in page["links"]:
            link, _ = urldefrag(link)
            if link in self.url_depths or not self.is_allowed(link):
                continue
            self.url_depths[link] = depth + 1
            self.queue.add_url(link, priority=-(depth + 1))

    async def close(self):
        await self.session.close()

## Демонстрация 1 дня

In [10]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

21:41:19 [INFO] Начало загрузки: https://example.com
21:41:20 [INFO] Успешно: https://example.com (статус 200, 577 символов)
21:41:20 [INFO] Начало загрузки: https://www.python.org
21:41:21 [INFO] Успешно: https://www.python.org (статус 200, 51758 символов)
21:41:21 [INFO] Начало загрузки: https://httpbin.org/get
21:41:22 [INFO] Успешно: https://httpbin.org/get (статус 200, 322 символов)
21:41:22 [INFO] Начало загрузки: https://httpbin.org/html
21:41:22 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)
21:41:22 [INFO] Начало загрузки: https://httpbin.org/delay/1
21:41:23 [INFO] Успешно: https://httpbin.org/delay/1 (статус 200, 372 символов)
21:41:23 [INFO] Начало загрузки: https://httpbin.org/delay/2
21:41:25 [INFO] Успешно: https://httpbin.org/delay/2 (статус 200, 372 символов)
21:41:25 [INFO] Начало загрузки: https://httpbin.org/delay/3
21:41:28 [INFO] Успешно: https://httpbin.org/delay/3 (статус 200, 372 символов)
21:41:28 [INFO] Начало загрузки: https://httpbin.o


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 9.77 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 3.92 сек

Параллельно быстрее в 2.5 раза


## Тестирование 1 дня

In [11]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_tests(tests: list):
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_tests([test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster])

21:41:34 [ERROR] ClientConnectorDNSError, сетевая ошибка: https://this-site-does-not-exist-12345.com


✅ test_valid_urls


21:41:34 [ERROR] ClientResponseError, статус 404: https://httpbin.org/status/404


✅ test_invalid_urls


21:41:37 [ERROR] SocketTimeoutError, таймаут: https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4


## Демонстрация 2 дня

In [12]:
DEMO_PAGES = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/html",
    "https://docs.python.org/3/",
]


def summarize(page: dict) -> dict:
    return {
        "url": page["url"],
        "title": page["title"],
        "h1": page["headings"].get("h1", []),
        "text_length": len(page["text"]),
        "links_count": len(page["links"]),
        "links": page["links"][:5],
        "images_count": len(page["images"]),
    }


async def demo_parsing(urls: list[str]):
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        pages = await asyncio.gather(*(crawler.fetch_and_parse(url) for url in urls))
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start

    for page in pages:
        print(json.dumps(summarize(page), ensure_ascii=False, indent=2))

    total_links = sum(len(page["links"]) for page in pages)
    total_text = sum(len(page["text"]) for page in pages)
    print("\nСтатистика")
    print(f"  Страниц: {len(pages)}")
    print(f"  Всего ссылок: {total_links}")
    print(f"  Общая длина текста: {total_text} символов")
    print(f"  Время: {elapsed:.2f} сек")


await demo_parsing(DEMO_PAGES)

21:41:48 [INFO] Начало загрузки: https://example.com
21:41:48 [INFO] Начало загрузки: https://www.python.org
21:41:48 [INFO] Начало загрузки: https://httpbin.org/html
21:41:48 [INFO] Начало загрузки: https://docs.python.org/3/
21:41:48 [INFO] Успешно: https://www.python.org (статус 200, 51758 символов)
21:41:48 [INFO] Успешно: https://docs.python.org/3/ (статус 200, 19620 символов)
21:41:49 [INFO] Успешно: https://example.com (статус 200, 577 символов)
21:41:49 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)


{
  "url": "https://example.com",
  "title": "Example Domain",
  "h1": [],
  "text_length": 171,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://www.python.org",
  "title": "Welcome to Python.org",
  "h1": [
    "Functions Defined",
    "All the Flow You’d Expect",
    "Compound Data Types",
    "Intuitive Interpretation",
    "Quick & Easy to Learn"
  ],
  "text_length": 6504,
  "links_count": 212,
  "links": [
    "https://www.python.org#content",
    "https://www.python.org#python-network",
    "https://www.python.org/",
    "https://www.python.org/psf/",
    "https://docs.python.org"
  ],
  "images_count": 1
}
{
  "url": "https://httpbin.org/html",
  "title": "",
  "h1": [
    "Herman Melville - Moby-Dick"
  ],
  "text_length": 3594,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://docs.python.org/3/",
  "title": "3.15.0 Documentation",
  "h1": [
    "Python 3.15.0 documentation"
  ],
  "text_length": 3217,
  "links_count": 9

## Тестирование 2 дня

In [13]:
VALID_HTML = """
<html>
  <head>
    <title>Тестовая страница</title>
    <meta name="description" content="Описание страницы">
  </head>
  <body>
    <h1>Главный заголовок</h1>
    <p>Основной текст страницы.</p>
    <img src="/logo.png" alt="Логотип">
    <table><tr><th>Город</th></tr><tr><td>Москва</td></tr></table>
    <ul><li>Чай</li><li>Кофе</li></ul>
  </body>
</html>
"""


async def test_parse_valid_html():
    result = await HTMLParser().parse_html(VALID_HTML, "https://example.com")
    assert result["title"] == "Тестовая страница", f"title: {result['title']}"
    assert result["metadata"]["description"] == "Описание страницы", "нет description"
    assert result["headings"]["h1"] == ["Главный заголовок"], "неверный h1"
    assert "Основной текст страницы." in result["text"], "нет текста"
    assert result["images"] == [
        {"src": "https://example.com/logo.png", "alt": "Логотип"}
    ], "неверные картинки"
    assert result["tables"] == [[["Город"], ["Москва"]]], "неверная таблица"
    assert result["lists"] == [{"type": "ul", "items": ["Чай", "Кофе"]}], "неверный список"


async def test_parse_broken_html():
    parser = HTMLParser()
    broken = "<title>Битая</title><h1>Заголовок<p>Текст<a href='/page'>ссылка<table><tr><td>A<td>B"
    result = await parser.parse_html(broken, "https://example.com")
    assert result["title"] == "Битая", f"title: {result['title']}"
    assert result["links"] == ["https://example.com/page"], f"links: {result['links']}"
    assert result["tables"] == [[["A", "B"]]], f"tables: {result['tables']}"

    result = await parser.parse_html(None, "https://example.com")
    assert result == parser.empty_result("https://example.com"), "ожидали пустой результат"


async def test_extract_links():
    html = """
    <a href="https://python.org">1</a>
    <a href="mailto:info@example.com">2</a>
    <a href="javascript:void(0)">3</a>
    <a href="tel:+70000000000">4</a>
    <a>без href</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com")
    assert links == ["https://python.org"], f"links: {links}"


async def test_relative_urls():
    html = """
    <a href="/about">1</a>
    <a href="page2.html">2</a>
    <a href="../archive">3</a>
    <a href="//cdn.example.org/lib">4</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com/news/today")
    assert links == [
        "https://example.com/about",
        "https://example.com/news/page2.html",
        "https://example.com/archive",
        "https://cdn.example.org/lib",
    ], f"links: {links}"


await run_tests([test_parse_valid_html, test_parse_broken_html, test_extract_links, test_relative_urls])

21:41:49 [WARNING] Не удалось разобрать HTML (TypeError): https://example.com


✅ test_parse_valid_html
✅ test_parse_broken_html
✅ test_extract_links
✅ test_relative_urls

Пройдено 4 из 4


## Демонстрация 3 дня

In [14]:
async def save_results(crawler: AsyncCrawler, path: str):
    data = {
        "processed": [
            {**page, "depth": crawler.url_depths[url]}
            for url, page in crawler.processed_urls.items()
        ],
        "failed": crawler.failed_urls,
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(data, ensure_ascii=False, indent=2))


async def demo_crawl():
    crawler = AsyncCrawler(max_concurrent=10, max_depth=2, per_domain_limit=3)
    logger.setLevel(logging.WARNING)
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://quotes.toscrape.com/",
                "https://books.toscrape.com/",
            ],
            max_pages=150,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()

    await save_results(crawler, "crawl_results.json")

    by_domain = Counter(urlparse(url).netloc for url in results)
    by_depth = Counter(crawler.url_depths[url] for url in results)
    print(f"\nОбработано: {len(results)} страниц")
    print(f"Ошибок: {len(crawler.failed_urls)}")
    print(f"По сайтам: {dict(by_domain)}")
    print(f"По глубине: {dict(sorted(by_depth.items()))}")
    print("Результаты сохранены в crawl_results.json")


await demo_crawl()

Обработано: 1 | В очереди: 37 | Ошибок: 0 | Скорость: 1.0 стр/сек
Обработано: 6 | В очереди: 107 | Ошибок: 0 | Скорость: 3.0 стр/сек
Обработано: 19 | В очереди: 147 | Ошибок: 0 | Скорость: 6.3 стр/сек
Обработано: 26 | В очереди: 155 | Ошибок: 0 | Скорость: 6.5 стр/сек
Обработано: 33 | В очереди: 161 | Ошибок: 0 | Скорость: 6.6 стр/сек
Обработано: 45 | В очереди: 178 | Ошибок: 0 | Скорость: 7.5 стр/сек
Обработано: 59 | В очереди: 313 | Ошибок: 0 | Скорость: 8.4 стр/сек
Обработано: 66 | В очереди: 404 | Ошибок: 0 | Скорость: 8.2 стр/сек
Обработано: 74 | В очереди: 501 | Ошибок: 0 | Скорость: 8.2 стр/сек
Обработано: 84 | В очереди: 590 | Ошибок: 0 | Скорость: 8.4 стр/сек
Обработано: 96 | В очереди: 622 | Ошибок: 0 | Скорость: 8.7 стр/сек
Обработано: 104 | В очереди: 619 | Ошибок: 0 | Скорость: 8.7 стр/сек
Обработано: 114 | В очереди: 609 | Ошибок: 0 | Скорость: 8.8 стр/сек
Обработано: 127 | В очереди: 598 | Ошибок: 0 | Скорость: 9.1 стр/сек
Обработано: 135 | В очереди: 590 | Ошибок: 0 | С

## Тестирование 3 дня

In [15]:
FAKE_SITE = {
    "https://site.com/": '<a href="/a">a</a><a href="/b">b</a><a href="https://other.com/x">x</a>',
    "https://site.com/a": '<a href="/a/deep">deep</a><a href="/">home</a><a href="/b">b</a>',
    "https://site.com/b": '<a href="/a">a</a><a href="/login">login</a>',
    "https://site.com/a/deep": '<a href="/a/deeper">deeper</a>',
    "https://site.com/a/deeper": "",
    "https://site.com/login": "",
    "https://other.com/x": "",
}


class FakeCrawler(AsyncCrawler):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.fetch_calls: list[str] = []

    async def fetch_url(self, url: str) -> str | None:
        self.fetch_calls.append(url)
        await asyncio.sleep(0.01)
        if url in FAKE_SITE:
            return f"<html><body>{FAKE_SITE[url]}</body></html>"
        self.failed_urls[url] = "NotFound"
        return None


async def run_fake_crawl(max_depth: int, **filters) -> FakeCrawler:
    crawler = FakeCrawler(max_concurrent=3, max_depth=max_depth)
    try:
        await crawler.crawl(["https://site.com/"], max_pages=100, **filters)
    finally:
        await crawler.close()
    return crawler


async def test_queue_priority():
    queue = CrawlerQueue()
    queue.add_url("https://site.com/a", priority=0)
    queue.add_url("https://site.com/b", priority=5)
    queue.add_url("https://site.com/c", priority=0)
    queue.add_url("https://site.com/d", priority=-1)
    queue.add_url("https://site.com/b", priority=10)

    order = [await queue.get_next() for _ in range(5)]
    assert order == [
        "https://site.com/b",
        "https://site.com/a",
        "https://site.com/c",
        "https://site.com/d",
        None,
    ], f"порядок: {order}"


async def test_depth_limit():
    crawler = await run_fake_crawl(max_depth=1)
    depths = [crawler.url_depths[url] for url in crawler.visited_urls]
    assert max(depths) == 1, f"глубины: {depths}"
    assert "https://site.com/a/deep" not in crawler.visited_urls, "зашли глубже max_depth"

    crawler = await run_fake_crawl(max_depth=2)
    assert "https://site.com/a/deep" in crawler.visited_urls, "не дошли до глубины 2"
    assert "https://site.com/a/deeper" not in crawler.visited_urls, "зашли на глубину 3"


async def test_url_filters():
    crawler = await run_fake_crawl(
        max_depth=3, same_domain_only=True, exclude_patterns=[r"/login"]
    )
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/b",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"

    crawler = await run_fake_crawl(max_depth=3, include_patterns=[r"/a"])
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"


async def test_no_duplicates():
    crawler = await run_fake_crawl(max_depth=3)
    calls = crawler.fetch_calls
    assert len(calls) == len(set(calls)), f"повторные загрузки: {calls}"
    assert set(calls) == crawler.visited_urls, "visited_urls не совпадает с загрузками"
    assert len(crawler.visited_urls) == len(FAKE_SITE), f"посещено {len(crawler.visited_urls)}"


await run_tests([test_queue_priority, test_depth_limit, test_url_filters, test_no_duplicates])


✅ test_queue_priority
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Скорость: 39.4 стр/сек
Обработано: 6 | В очереди: 0 | Ошибок: 0 | Скорость: 59.3 стр/сек
✅ test_depth_limit
Обработано: 5 | В очереди: 0 | Ошибок: 0 | Скорость: 49.3 стр/сек
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Скорость: 39.5 стр/сек
✅ test_url_filters
Обработано: 7 | В очереди: 0 | Ошибок: 0 | Скорость: 69.1 стр/сек
✅ test_no_duplicates

Пройдено 4 из 4
